# Run the BESSTIE deployment on Colab (GPU)

This notebook runs Mohamed's Gradio app (`app/app.py` on the `main` branch) on a Colab T4 GPU and exposes a public URL via Gradio's built-in tunnel. Use it for:

- Capturing the screenshots required for §5.1 of the report.
- Demoing the app to teammates without each of them needing local Python.
- Running latency benchmarks (Q5.2) on a GPU instead of a Mac CPU.

**Setup checklist before running:**

1. `Runtime → Change runtime type → T4 GPU` (free tier is enough; an A100 if you have Pro+).
2. Run the cells top-to-bottom.
3. The last cell prints a `https://...gradio.live` URL that anyone can open.

## 1. Sanity check — GPU and Python

In [ ]:
!nvidia-smi || echo 'No GPU detected. Switch the runtime to T4 GPU first.'
import sys; print('Python:', sys.version)

## 2. Clone the repo

We pull the `main` branch of the team repo, which contains `app/app.py`. If you're testing pre-merge changes, replace `--branch main` with whichever branch has the latest version of the app.

In [ ]:
import os, subprocess, sys

# Public mirror of Fiyin's pipeline branch — clones without auth on Colab.
REPO_URL = os.environ.get('REPO_URL', 'https://github.com/TheFinix13/NLP-coursework.git')
BRANCH   = os.environ.get('REPO_BRANCH', 'main')
REPO_DIR = '/content/NLP-coursework'

if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.chdir(REPO_DIR)
!ls app/

## 3. Install dependencies

Colab already ships with `torch` and `transformers`; we add `peft`, the right `gradio` version, and a couple of small helpers.

In [ ]:
!pip install -q --upgrade peft 'gradio>=4.0' accelerate

## 4. Patch the app to expose a public URL

Mohamed's `app/app.py` ends with `demo.launch()`. On Colab we want `demo.launch(share=True)` so Gradio gives us a `*.gradio.live` URL that the team and the report screenshots can reach. We patch the file in-place — this only modifies the Colab copy, not the repo on GitHub.

In [ ]:
with open('app/app.py', 'r') as f:
    src = f.read()

if 'demo.launch(share=True' not in src:
    src = src.replace('demo.launch()', 'demo.launch(share=True, server_name="0.0.0.0")')
    with open('app/app.py', 'w') as f:
        f.write(src)

print(src.splitlines()[-1])

## 5. Run the app

First boot pulls `facebook/opt-1.3b` (~2.6 GB) and the three small LoRA adapters. On a T4 this is ~30–60 s end-to-end. Once it prints the `Running on public URL: https://...gradio.live` line, open that URL in your browser.

The app keeps running while this cell is alive; **do not stop the cell** until you're done with it. Use the menu button on the cell to stop the kernel when you're finished.

In [ ]:
!python app/app.py

## 6. (Optional) Smoke-test sentences

Paste these into the **Compare all adapters** tab to see the three LoRA adapters disagree (the screenshot for `[FIGURE 5.1.3]` in `reports/results/q5_1_deployment.md`):

```
Absolute legend, parked his ute right across my driveway. Good onya, mate.
Coz we all have free internet.
Cheerful fellow aren't you.
What a brave potatriot
The Interior was Too Good and Test Was Awesome.
```

Use macOS `Cmd+Shift+4` (or Windows `Win+Shift+S`) to capture, save into `reports/figures/q5_1_screenshot_*.png`, and reference in the docx.

## 7. (Optional) Run the latency benchmark on the same GPU

While the app is running, you can open a second cell and run the Q5.2 benchmark to grab the latency numbers for the report. Stop the app cell first or run on a fresh runtime — both processes share the same VRAM.

In [ ]:
# Run only after the app cell is stopped (otherwise both fight for VRAM).
# !python scripts/benchmark_inference.py \
#     --tfidf-vec  notebooks/models/tfidf/tfidf_vectorizer.pkl \
#     --tfidf-clf  notebooks/models/LogisticRegression_sarcasm.pkl \
#     --roberta    roberta-base \
#     --base-llm   facebook/opt-1.3b \
#     --lora       momofahmi/besstie-lora-en-uk-opt-1.3b \
#     --out        reports/results/q5_2_efficiency.json